## 🎯 **Obiettivo di questo esercizio**

Imparare a **"conoscere"** una tabella prima di toccarla:
- Quante righe? Quante colonne?
- Che tipi di dati? Quanti NULL?
- Quali pattern di sporcizia esistono?
- Quali valori anomali (outlier)?
---
## 📋 **Metodi e funzioni (riassunto)**

### **In SQLite (esplorazione)**

| Funzione | Cosa fa | Esempio |
|----------|---------|---------|
| `SELECT COUNT(*)` | Numero righe | `SELECT COUNT(*) FROM tabella;` |
| `PRAGMA table_info(tabella)` | Struttura colonne | `PRAGMA table_info(sondaggi);` |
| `SELECT DISTINCT colonna` | Valori unici | `SELECT DISTINCT Sito FROM sondaggi;` |
| `SELECT COUNT(DISTINCT colonna)` | Conteggio valori unici | `SELECT COUNT(DISTINCT Sito) FROM sondaggi;` |
| `colonna IS NULL` | Trovare NULL | `SELECT COUNT(*) FROM tabella WHERE colonna IS NULL;` |
| `MIN()`, `MAX()`, `AVG()` | Statistiche base | `SELECT MIN(profondità), MAX(profondità), AVG(profondità) FROM sondaggi;` |
| `colonna GLOB '[0-9]*'` | Pattern matching | Trova valori che iniziano con numeri |
| `colonna LIKE '%[a-z]%'` | Trova lettere minuscole | `WHERE Sito LIKE '%[a-z]%'` |

## 🎯 **Quesiti (esplorazione SOLO, nessuna pulizia)**

Usando **SQL** o **pandas** (scegli tu), rispondi a queste domande:

### **1. Panoramica generale**
- Quante righe ha la tabella?
- Quante colonne? Quali sono i loro nomi?
- Quali colonne sono numeriche? Quali sono testo?

### **2. Analisi NULL**
- Quanti NULL ci sono in `Au_ppm`?
- Quanti NULL in `Profondita_m`?
- Quale percentuale del totale rappresentano?

### **3. Pattern di sporcizia su ID_Campione**
- Quanti ID sono in MAIUSCOLO?
- Quanti in minuscolo?
- Quanti hanno spazi?
- Quanti hanno il trattino `-`?
- Quanti NON hanno il trattino?

### **4. Pattern di sporcizia su Sito**
- Elenca tutti i valori unici (compresi quelli sporchi)
- Quanti sono in MAIUSCOLO? Minuscolo?
- Quanti hanno spazi?

### **5. Analisi Profondita_m**
- Quali valori non sono numerici? (es. "zero", "ND", "mille")
- Quali sono outlier evidenti? (valori negativi o > 1000)
- Qual è la media, mediana, min, max dei valori puliti (escludendo outlier e non-numerici)?

### **6. Analisi Au_ppm**
- Quali valori non sono numerici? ("ND", "traccia")
- Quali sono outlier? (valori > 10 ppm)
- Distribuzione: media, mediana, min, max (escludendo outlier e non-numerici)

### **7. Analisi Data**
- Quanti formati di data diversi vedi?
- Elenca i valori unici della colonna Data
- Quali record hanno "oggi"?

---

## 📝 **Consegna**

Crea un notebook con:
1. Il codice per caricare il dataset
2. Le tue query/comandi di esplorazione
3. Per ogni domanda, scrivi:
   - Il codice usato
   - Il risultato ottenuto
   - Una breve nota su cosa quel risultato ti dice

**Non scrivere ancora codice di pulizia!** Solo esplorazione.

In [1]:
import sqlite3
import pandas as pd
import random
import numpy as np

conn = sqlite3.connect(':memory:')

# Genera 50 campioni con vari pattern di sporcizia
np.random.seed(42)
id_campione = [f"CAMP-{i:03d}" for i in range(1, 51)]
# Aggiungi sporcizia a caso
for i in range(len(id_campione)):
    if i % 7 == 0:
        id_campione[i] = id_campione[i].lower()
    elif i % 11 == 0:
        id_campione[i] = id_campione[i].replace("-", "")
    elif i % 13 == 0:
        id_campione[i] = " " + id_campione[i] + " "

siti = ["Nord", "Sud", "Est", "Ovest", "Centrale"]
sito = [random.choice(siti) for _ in range(50)]
# Aggiungi sporcizia
for i in range(len(sito)):
    if i % 5 == 0:
        sito[i] = sito[i].lower()
    elif i % 17 == 0:
        sito[i] = sito[i] + " "

profondita = [round(random.uniform(10, 300), 1) for _ in range(50)]
# Aggiungi valori anomali e sporcizia
profondita[3] = "zero"
profondita[12] = "ND"
profondita[25] = -50  # outlier
profondita[38] = "mille"
profondita[42] = 9999  # outlier

au_ppm = [round(random.uniform(0, 5), 2) for _ in range(50)]
au_ppm[7] = "ND"
au_ppm[14] = "traccia"
au_ppm[28] = 99.9  # outlier estremo
au_ppm[35] = None

data = [f"2024-{random.randint(1,12):02d}-{random.randint(1,28):02d}" for _ in range(50)]
# Aggiungi formati data sporchi
data[5] = "15/06/2024"
data[19] = "2024.09.20"
data[33] = "20-10-2024"
data[47] = "oggi"

# Crea DataFrame
df = pd.DataFrame({
    'ID_Campione': id_campione,
    'Sito': sito,
    'Profondita_m': profondita,
    'Au_ppm': au_ppm,
    'Data': data
})

df.to_sql('campioni', conn, index=False, if_exists='replace')

print("✅ Dataset creato! 50 righe con sporcizia realistica.")
print("\nPrime 10 righe:")
print(df.head(10))
print(f"\nShape: {df.shape}")

✅ Dataset creato! 50 righe con sporcizia realistica.

Prime 10 righe:
  ID_Campione      Sito Profondita_m Au_ppm        Data
0    camp-001       est        122.3   3.83  2024-08-25
1    CAMP-002       Est         43.1    3.6  2024-07-03
2    CAMP-003       Est         39.6   0.34  2024-08-24
3    CAMP-004       Est         zero   0.41  2024-08-18
4    CAMP-005      Nord         58.5   4.64  2024-10-21
5    CAMP-006  centrale        147.3   1.69  15/06/2024
6    CAMP-007     Ovest        217.1    3.5  2024-05-22
7    camp-008       Est         83.2     ND  2024-04-20
8    CAMP-009      Nord        197.5   4.92  2024-05-07
9    CAMP-010      Nord        194.5   1.89  2024-04-11

Shape: (50, 5)
